# **Причинные графы: d-разделение, бэкдор, фронтдор и медиация**

Практика к модулю [«Каузальность и линейные модели»](https://ai-interpretability.school), урок «Графы: d-разделение, бэкдор и фронтдор».

В уроке граф был картинкой со стрелками, а правила — формулами. Здесь мы зададим граф структурными уравнениями, сгенерируем по нему данные и проверим на них четыре утверждения урока:

1. d-разделение в графе означает условную независимость в данных, и критерий бэкдора выбирает набор контроля без перебора наугад;
2. переменная, которая одновременно конфаундер и коллайдер, в одиночку смещает оценку, хотя «похожа на правильный контроль»;
3. формула фронтдора восстанавливает эффект, когда общая причина воздействия и исхода не измерена;
4. полный эффект раскладывается на прямой и косвенный, и раскладка ломается, когда у медиатора и исхода есть скрытая общая причина.

План тетради:

1. граф и данные по структурным уравнениям;
2. d-разделение и частные корреляции;
3. критерий бэкдора: все допустимые наборы контроля;
4. формула корректировки и оценки эффекта при разных наборах;
5. фронтдор на бинарных данных со скрытым конфаундером;
6. прямой и косвенный эффект;
7. где ломается: медиатор и коллайдер в контролях, скрытый конфаундер медиатора;
8. те же расчеты в DoWhy.

Все данные синтетические: только так истинный эффект известен и оценку есть с чем сравнить. Тетрадь идет на процессоре, прогон занимает около минуты.

In [ ]:
%pip install -q dowhy==0.14

In [ ]:
import itertools
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.mediation import Mediation

warnings.filterwarnings('ignore')
SEED = 0
pd.set_option('display.width', 140)

## 1. Граф и данные

Структурная причинная модель (SCM) задает каждую переменную уравнением от ее родителей в графе и собственного шума. Пример — тот же, что в уроке: курс повышения квалификации $T$ (часы обучения) и зарплата $Y$.

- $X_1$ — опыт работы, $X_2$ — сеть профессиональных знакомств; друг от друга они не зависят;
- $X_3$ — оценка руководительницы: зависит и от опыта, и от знакомств, а сама влияет на то, сколько часов обучения дают сотруднице, и на зарплату;
- $M$ — навык, который дает курс; $C$ — приглашение на собеседование в другую компанию, следствие и курса, и зарплаты.

$$X_1=\varepsilon_1,\quad X_2=\varepsilon_2,\quad X_3=0.8X_1+0.8X_2+\varepsilon_3,\quad T=0.7X_1+0.6X_3+\varepsilon_T,$$
$$M=0.9T+\varepsilon_M,\quad Y=0.5T+0.8M+0.7X_2+0.6X_3+\varepsilon_Y,\quad C=0.6T+0.8Y+\varepsilon_C,$$

где все шумы $\varepsilon$ независимы и распределены как $\mathcal N(0,1)$.

Вмешательство $do(T=t)$ заменяет уравнение для $T$ константой и оставляет остальные уравнения как есть. Поэтому истинный полный эффект часа обучения — сумма произведений коэффициентов по направленным путям из $T$ в $Y$: $0.5 + 0.9\cdot 0.8 = 1.22$. Прямой эффект — $0.5$, косвенный, через навык, — $0.72$.

In [ ]:
G = nx.DiGraph([('X1', 'T'), ('X1', 'X3'), ('X2', 'X3'), ('X2', 'Y'), ('X3', 'T'), ('X3', 'Y'),
                ('T', 'M'), ('M', 'Y'), ('T', 'Y'), ('T', 'C'), ('Y', 'C')])

def simulate(n, seed=SEED):
    rng = np.random.default_rng(seed)
    e = lambda: rng.normal(size=n)
    X1 = e(); X2 = e()
    X3 = 0.8 * X1 + 0.8 * X2 + e()
    T = 0.7 * X1 + 0.6 * X3 + e()
    M = 0.9 * T + e()
    Y = 0.5 * T + 0.8 * M + 0.7 * X2 + 0.6 * X3 + e()
    C = 0.6 * T + 0.8 * Y + e()
    return pd.DataFrame(dict(X1=X1, X2=X2, X3=X3, T=T, M=M, Y=Y, C=C))

df = simulate(20000)
TRUE_TOTAL, TRUE_DIRECT, TRUE_INDIRECT = 0.5 + 0.9 * 0.8, 0.5, 0.9 * 0.8
print('объектов:', len(df), '| истинный полный эффект:', round(TRUE_TOTAL, 2))
df.head().round(2)

## 2. d-разделение и частные корреляции

Путь между двумя вершинами блокирован набором $S$, если на нем есть цепочка $A\to B\to C$ или развилка $A\leftarrow B\to C$ с серединой $B$ в $S$, или коллайдер $A\to B\leftarrow C$, у которого ни $B$, ни его потомки не входят в $S$. Вершины d-разделены набором $S$, если блокированы все пути между ними. В `networkx` это функция `nx.is_d_separator(G, x, y, S)`.

В линейной модели с гауссовым шумом условная независимость — это нулевая частная корреляция: корреляция остатков двух переменных после регрессии каждой на набор $S$. Сравним вердикт графа с данными для нескольких пар.

In [ ]:
def partial_corr(data, a, b, S):
    if not S:
        return np.corrcoef(data[a], data[b])[0, 1]
    Z = sm.add_constant(data[list(S)])
    ra = sm.OLS(data[a], Z).fit().resid
    rb = sm.OLS(data[b], Z).fit().resid
    return np.corrcoef(ra, rb)[0, 1]

checks = [('X1', 'X2', []), ('X1', 'X2', ['X3']), ('X1', 'Y', ['X3', 'T']), ('X1', 'Y', ['X2', 'X3', 'T']),
          ('T', 'X2', ['X3']), ('T', 'X2', ['X1', 'X3']), ('M', 'X3', ['T'])]
rows = [dict(пара=f'{a} и {b}', набор=', '.join(S) or '—', d_разделены=nx.is_d_separator(G, {a}, {b}, set(S)),
             частная_корреляция=round(partial_corr(df, a, b, S), 3)) for a, b, S in checks]
pd.DataFrame(rows)

Где граф говорит «d-разделены», частная корреляция в пределах шума выборки (на 20 000 объектах это сотые доли); где «не разделены» — заметно отличается от нуля. Вторая строка — коллайдер $X_3$: опыт и знакомства независимы, но становятся связаны, как только мы фиксируем оценку руководительницы. Связь с одинаковыми коэффициентами $0.8$ получается отрицательной: при той же оценке у сотрудницы с большим опытом знакомств в среднем меньше.

## 3. Критерий бэкдора

Набор $S$ удовлетворяет критерию бэкдора относительно пары $(T, Y)$, если

1. в $S$ нет потомков $T$;
2. $S$ блокирует все пути из $T$ в $Y$, которые начинаются стрелкой, входящей в $T$.

Второе условие проверяется d-разделением в графе, из которого убраны все стрелки, выходящие из $T$: в таком графе остались только «черные ходы». Переберем все подмножества пяти измеренных переменных $X_1, X_2, X_3, M, C$.

In [ ]:
def satisfies_backdoor(G, t, y, S):
    if set(S) & nx.descendants(G, t):
        return False
    G_back = G.copy()
    G_back.remove_edges_from(list(G.out_edges(t)))
    return nx.is_d_separator(G_back, {t}, {y}, set(S))

candidates = ['X1', 'X2', 'X3', 'M', 'C']
subsets = [S for k in range(len(candidates) + 1) for S in itertools.combinations(candidates, k)]
valid = [S for S in subsets if satisfies_backdoor(G, 'T', 'Y', S)]
print('подмножеств всего:', len(subsets))
print('допустимых наборов:', len(valid), valid)
print('минимальный набор (networkx):', nx.find_minimal_d_separator(
    nx.restricted_view(G, [], list(G.out_edges('T'))), 'T', 'Y', restricted={'X1', 'X2', 'X3'}))

**Задание 1. Сколько наборов из 32 подмножеств $\{X_1, X_2, X_3, M, C\}$ удовлетворяют критерию бэкдора? Почему набор из одной $X_3$ в их число не входит, хотя $X_3$ — общая причина $T$ и $Y$?**

## 4. Формула корректировки

Если набор $S$ удовлетворяет критерию бэкдора, то

$$P(y\mid do(t))=\sum_s P(y\mid t,s)\,P(s).$$

В линейной модели без взаимодействий сумма сворачивается: эффект единицы $T$ — коэффициент при $T$ в регрессии $Y$ на $T$ и $S$. Посчитаем его для пустого набора, для каждого допустимого, для одной $X_3$ и для наборов с медиатором и коллайдером.

In [ ]:
def adjusted_effect(data, S):
    formula = 'Y ~ T' + ''.join(' + ' + s for s in S)
    return smf.ols(formula, data=data).fit().params['T']

sets = [(), ('X3',)] + valid + [('X2', 'X3', 'M'), ('X2', 'X3', 'C')]
table = pd.DataFrame([dict(набор=', '.join(S) or '—', бэкдор=satisfies_backdoor(G, 'T', 'Y', S),
                           оценка=round(adjusted_effect(df, S), 3)) for S in sets])
table['ошибка'] = (table['оценка'] - TRUE_TOTAL).round(3)
table

**Задание 2. Чему равна оценка эффекта при наборе $\{X_2, X_3\}$ (три знака после точки)? Насколько она отличается от оценки при наборе из одной $X_3$, и в какую сторону смещена оценка без контроля?**

Все три допустимых набора дают оценки в пределах шума от истинных $1.22$. Одна $X_3$ закрывает развилки $T\leftarrow X_3\to Y$ и $T\leftarrow X_1\to X_3\to Y$, но открывает путь $T\leftarrow X_1\to X_3\leftarrow X_2\to Y$, где $X_3$ — коллайдер. Медиатор в контролях отрезает косвенный эффект, и остается прямой, около $0.5$; коллайдер в контролях смещает оценку сильнее всех.

## 5. Фронтдор: скрытый конфаундер

Новый граф: склонность к саморазвитию $U$ (в данных ее нет) влияет и на то, пойдет ли сотрудница на курс $T$, и на повышение $Y$; курс дает сертификат $M$, а повышение зависит от сертификата, но не от самого курса. Все переменные бинарные:

$$U\sim\text{Bern}(0.5),\quad T\sim\text{Bern}(0.2+0.6U),\quad M\sim\text{Bern}(0.1+0.7T),\quad Y\sim\text{Bern}(0.1+0.5M+0.3U).$$

Истинный эффект: $P(Y=1\mid do(T=1))-P(Y=1\mid do(T=0)) = 0.5\cdot 0.7 = 0.35$. Бэкдор здесь не применим: единственный «черный ход» $T\leftarrow U\to Y$ идет через неизмеренную $U$. Но $M$ удовлетворяет критерию фронтдора, и

$$P(y\mid do(t))=\sum_m P(m\mid t)\sum_{t'}P(y\mid m,t')\,P(t').$$

In [ ]:
rng = np.random.default_rng(SEED)
n_fd = 20000
U = rng.binomial(1, 0.5, n_fd)
T_fd = rng.binomial(1, 0.2 + 0.6 * U)
M_fd = rng.binomial(1, 0.1 + 0.7 * T_fd)
Y_fd = rng.binomial(1, 0.1 + 0.5 * M_fd + 0.3 * U)
fd = pd.DataFrame(dict(T=T_fd, M=M_fd, Y=Y_fd))   # U в таблицу не попадает
TRUE_FD = 0.5 * 0.7

naive_fd = fd.loc[fd['T'] == 1, 'Y'].mean() - fd.loc[fd['T'] == 0, 'Y'].mean()
print('наивная разность долей:', round(naive_fd, 3), '| истинный эффект:', TRUE_FD)

In [ ]:
p_t = fd['T'].value_counts(normalize=True).sort_index()   # P(t')
p_m1_t = fd.groupby('T')['M'].mean()                      # P(M=1 | t)
p_y1_mt = fd.groupby(['M', 'T'])['Y'].mean()              # P(Y=1 | m, t')

def p_y1_do(t):
    total = 0.0
    for m in (0, 1):
        p_m = p_m1_t[t] if m == 1 else 1 - p_m1_t[t]
        total += p_m * sum(p_y1_mt[(m, tp)] * p_t[tp] for tp in (0, 1))
    return total

frontdoor = p_y1_do(1) - p_y1_do(0)
print('P(Y=1 | do(T=1)) =', round(p_y1_do(1), 3), '| P(Y=1 | do(T=0)) =', round(p_y1_do(0), 3))
print('оценка по фронтдору:', round(frontdoor, 3))

**Задание 3. Чему равна оценка эффекта курса по формуле фронтдора (три знака после точки)? Во сколько раз наивная разность долей больше истинного эффекта $0.35$?**

Внутренняя сумма по $t'$ — та же формула корректировки, только для пары $(M, Y)$: путь $M\leftarrow T\leftarrow U\to Y$ закрывается контролем по $T$. Внешняя сумма переносит эффект $T$ на $M$, который не смешан, потому что у $T$ и $M$ общих причин нет.

## 6. Прямой и косвенный эффект

Вернемся к первому графу. В линейной модели без взаимодействия $T$ и $M$ косвенный эффект — произведение коэффициентов $a\cdot b$ ([Baron и Kenny, 1986](https://doi.org/10.1037/0022-3514.51.6.1173)): $a$ — коэффициент при $T$ в регрессии $M$ на $T$ и контроли, $b$ — коэффициент при $M$ в регрессии $Y$ на $T$, $M$ и контроли. Прямой эффект — коэффициент при $T$ во второй регрессии, и в сумме они дают полный эффект. Контроли — допустимый набор $\{X_2, X_3\}$: он закрывает и «черные ходы» $T$, и общие причины $M$ и $Y$ кроме $T$.

`statsmodels.stats.mediation.Mediation` считает то же через симуляцию по алгоритму [Imai, Keele и Tingley (2010)](https://doi.org/10.1037/a0020761): ACME — средний косвенный эффект, ADE — средний прямой.

In [ ]:
def product_of_coefficients(data, controls):
    c = ''.join(' + ' + s for s in controls)
    a = smf.ols('M ~ T' + c, data=data).fit().params['T']
    out = smf.ols('Y ~ T + M' + c, data=data).fit()
    return a, out.params['M'], out.params['T']

a, b, direct = product_of_coefficients(df, ['X2', 'X3'])
print(f'a = {a:.3f}, b = {b:.3f}')
print(f'косвенный a*b = {a * b:.3f} (истинный {TRUE_INDIRECT:.2f}), прямой = {direct:.3f} (истинный {TRUE_DIRECT})')
print(f'сумма = {a * b + direct:.3f}, доля косвенного = {a * b / (a * b + direct):.3f}')

med = Mediation(sm.OLS.from_formula('Y ~ T + M + X2 + X3', df), sm.OLS.from_formula('M ~ T + X2 + X3', df),
                'T', 'M').fit(n_rep=200, rng=np.random.default_rng(SEED))
med.summary().loc[['ACME (average)', 'ADE (average)', 'Total effect', 'Prop. mediated (average)']].round(3)

**Задание 4. Чему равен косвенный эффект $a\cdot b$ при контролях $\{X_2, X_3\}$ (три знака после точки)? Какая доля полного эффекта идет через навык?**

## 7. Где ломается

### 7.1. Медиатор и коллайдер в контролях

Таблица раздела 4 уже показала оба случая: с медиатором в контролях регрессия оценивает прямой эффект вместо полного, с коллайдером — величину, которая не равна ни тому, ни другому. Метрика качества регрессии при этом растет: $C$ — следствие $Y$ и хорошо его предсказывает.

In [ ]:
for S in [('X2', 'X3'), ('X2', 'X3', 'M'), ('X2', 'X3', 'C')]:
    fit = smf.ols('Y ~ T' + ''.join(' + ' + s for s in S), data=df).fit()
    print(f'{", ".join(S):12s} оценка {fit.params["T"]:.3f}   R^2 {fit.rsquared:.3f}')

### 7.2. Скрытая общая причина медиатора и исхода

Разложение на прямой и косвенный эффект требует больше, чем полный эффект: кроме «черных ходов» $T$ не должно быть скрытых общих причин $M$ и $Y$ (у Imai и соавторов это допущение sequential ignorability). Добавим в первый граф мотивацию $V$, которая поднимает и навык, и зарплату, и не запишем ее в данные. Полный эффект от этого не меняется: $V$ не влияет на $T$.

In [ ]:
def simulate_hidden_mediator_confounder(n, strength, seed=SEED):
    rng = np.random.default_rng(seed)
    e = lambda: rng.normal(size=n)
    X1 = e(); X2 = e(); V = e()
    X3 = 0.8 * X1 + 0.8 * X2 + e()
    T = 0.7 * X1 + 0.6 * X3 + e()
    M = 0.9 * T + strength * V + e()
    Y = 0.5 * T + 0.8 * M + 0.7 * X2 + 0.6 * X3 + strength * V + e()
    return pd.DataFrame(dict(X1=X1, X2=X2, X3=X3, T=T, M=M, Y=Y))

rows = []
for strength in [0.0, 0.5, 1.0]:
    d = simulate_hidden_mediator_confounder(20000, strength)
    a_v, b_v, direct_v = product_of_coefficients(d, ['X2', 'X3'])
    rows.append(dict(сила_V=strength, полный=round(adjusted_effect(d, ('X2', 'X3')), 3),
                     косвенный_ab=round(a_v * b_v, 3), прямой=round(direct_v, 3)))
pd.DataFrame(rows)

Полный эффект остается около $1.22$ при любой силе $V$, а раскладка уезжает: $b$ впитывает связь $M$ и $Y$ через $V$, косвенный эффект растет, прямой уходит вниз и при силе $1$ почти обнуляется. По данным это не видно: регрессии те же, что в разделе 6. Поэтому Imai, Keele и Tingley предлагают вместе с оценкой показывать анализ чувствительности к нарушению этого допущения.

## 8. Те же расчеты в DoWhy

[DoWhy](https://github.com/py-why/dowhy) (лицензия MIT) делит работу на те же шаги: граф → идентификация (какой критерий дает формулу) → оценка. `identify_effect` сам находит набор по критерию бэкдора, а для второго графа — формулу фронтдора; переменную `U` из графа он считает неизмеренной, потому что ее нет в таблице.

In [ ]:
from dowhy import CausalModel

model = CausalModel(data=df, treatment='T', outcome='Y', graph=G)
estimand = model.identify_effect()
print('набор по бэкдору:', estimand.get_backdoor_variables())
est = model.estimate_effect(estimand, method_name='backdoor.linear_regression')
print('DoWhy, бэкдор:', round(est.value, 3))

G_fd = nx.DiGraph([('U', 'T'), ('U', 'Y'), ('T', 'M'), ('M', 'Y')])
model_fd = CausalModel(data=fd, treatment='T', outcome='Y', graph=G_fd)
estimand_fd = model_fd.identify_effect()
print('фронтдор через:', estimand_fd.get_frontdoor_variables())
est_fd = model_fd.estimate_effect(estimand_fd, method_name='frontdoor.two_stage_regression')
print('DoWhy, фронтдор:', round(est_fd.value, 3), '| наша формула:', round(frontdoor, 3))

`frontdoor.two_stage_regression` в DoWhy — линейная версия формулы: коэффициент $T$ в регрессии $M$ на $T$, умноженный на коэффициент $M$ в регрессии $Y$ на $M$ и $T$. На бинарных данных это приближение, и оно может отличаться от точной суммы раздела 5 в последних знаках.

## Что унести из тетради

- **Граф решает, что контролировать, до всяких данных.** Критерий бэкдора проверяется d-разделением в графе без выходящих из $T$ стрелок; перебор подмножеств заменяет догадки «контролировать все, что связано с $T$».
- **Конфаундер может быть и коллайдером.** $X_3$ — общая причина $T$ и $Y$, но контроль по ней одной открывает путь через $X_1$ и $X_2$; помогает только набор, который закрывает и этот путь.
- **Фронтдор обходится без измерения конфаундера,** но требует, чтобы весь эффект шел через измеренный медиатор и чтобы у $T$ и $M$, а также у $M$ и $Y$ (кроме самой $T$) не было скрытых общих причин.
- **Прямой и косвенный эффект требуют больше допущений, чем полный.** Скрытая общая причина медиатора и исхода не трогает полный эффект, а раскладку искажает, и по данным этого не видно.

Источники:
1. [Pearl, 1995. Causal diagrams for empirical research](https://doi.org/10.1093/biomet/82.4.702)
2. [Pearl, 2001. Direct and Indirect Effects](https://arxiv.org/abs/1301.2300)
3. [Baron & Kenny, 1986. The moderator–mediator variable distinction in social psychological research](https://doi.org/10.1037/0022-3514.51.6.1173)
4. [Imai, Keele & Tingley, 2010. A general approach to causal mediation analysis](https://doi.org/10.1037/a0020761)
5. [Sharma & Kiciman, 2020. DoWhy: An End-to-End Library for Causal Inference](https://arxiv.org/abs/2011.04216)